# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a FAIR^2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is described in [Croissant](https://mlcommons.org/croissant/) format and accessible via the following schema URL.

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the provided Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define URL of the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and columns, and their unique `@id`s included in the FAIR^2 dataset.

Below, you'll see a summary of available record sets and the fields and columns they contain. *All record sets and fields are referenced by their `@id`.*

In [ ]:
# List all available record sets in the dataset, referencing by @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets are explicitly listed in the metadata. Attempting to infer available record sets from data files...")
    # Try to guess record sets from files (common with some Croissant schemas)
    # For demonstration, we'll obtain them by extracting the @id from distributions with known CSVs (if possible):
    all_distributions = getattr(metadata, 'distribution', [])
    print(f"Available distributions (potential record sets):")
    for dist in all_distributions:
        print(f"  - {dist['@id']}")
    # Manually set record set IDs for the example
    record_sets = [dist['@id'] for dist in all_distributions]
else:
    print("Record sets (by @id):")
    for rec in record_sets:
        print(f"  - {rec['@id']}")

# For each available record set, list available fields (columns) by @id
fields_by_record_set = {}
for record_set in record_sets:
    try:
        # Each record_set may provide fields (columns) meta info:
        rs = dataset.record_set(record_set=record_set)
        fields = getattr(rs, 'fields', [])
        field_ids = [f['@id'] for f in fields] if fields else []
    except Exception as e:
        field_ids = []
    fields_by_record_set[record_set] = field_ids
    print(f"\nFields/columns for record set {record_set}:")
    if field_ids:
        for fid in field_ids:
            print(f"  - {fid}")
    else:
        print("  [Field information not available in the metadata, will infer from data loading]")

## 3. Data Extraction
Here we extract data from each record set referenced via its `@id` and load it into separate pandas DataFrames. If field (column) `@id`s are unknown, we'll infer them from the loaded data.

In [ ]:
# Load records for each discovered record set
dataframes = {}

print("\nLoading data for each record set:\n")
for record_set_id in record_sets:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if not records:
            print(f"No records found for {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Failed to load data for {record_set_id}: {e}\n")

if not dataframes:
    raise RuntimeError("No record set dataframes could be loaded. Please review the record set IDs and data availability.")

# Pick the first available record set for demonstration
example_record_set = next(iter(dataframes))
print(f"\nUsing record set for further analysis: {example_record_set}\n")
print("Sample data from record set:")
print(dataframes[example_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Let's run common data processing and transformation steps. We'll select a numeric column (by its column `@id` if available), filter, and normalize values, and group by a categorical column (referenced by its `@id`).

In [ ]:
# Identify a numeric and a grouping column by @id if possible
df = dataframes[example_record_set]
numeric_field_candidates = [col for col in df.columns if df[col].dtype.kind in 'ifc' or (df[col].astype(str).str.replace('.', '', 1).str.isnumeric().all())]
if not numeric_field_candidates:
    # Fallback: select columns with numeric-like names
    numeric_field_candidates = [col for col in df.columns if 'coef' in col.lower() or 'value' in col.lower() or 'iteration' in col.lower() or 'likelihood' in col.lower()]

if not numeric_field_candidates:
    raise ValueError('No numeric fields found in the example record set.')

numeric_field_id = numeric_field_candidates[0]  # For reproducibility
print(f"Selected numeric field for EDA (by @id): {numeric_field_id}")

# Optional: select a grouping field
potential_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field_id = potential_group_fields[0] if potential_group_fields else None

print(f"Grouping field (by @id): {group_field_id}\n")

# Convert numeric column to numeric dtype if needed
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Example filtering: values > threshold
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id] + ([group_field_id] if group_field_id else [])].head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized values for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional grouping
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of our selected numeric field and explore its relationship to our grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a group field is available, plot boxplot
if group_field_id is not None:
    plt.figure(figsize=(10,5))
    # Truncate to 10 groups to avoid overplotting
    counts = df[group_field_id].value_counts()
    top_groups = counts.index[:10]
    sns.boxplot(x=df[group_field_id][df[group_field_id].isin(top_groups)], 
                y=df[numeric_field_id][df[group_field_id].isin(top_groups)])
    plt.xticks(rotation=45)
    plt.title(f'{numeric_field_id} by {group_field_id} (top 10)')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- The dataset schema allows loading of complex analytical survey results and model outputs using Croissant and `mlcroissant`.
- We explored available record sets (by `@id`), loaded their contents, and performed common EDA steps referenced by column `@id`.
- Through filtering and normalization of a selected numeric field, we demonstrated basic data processing. Grouping and visualization allow for further insight into factors associated with adoption of knowledge systems in rangeland intervention outcomes.

For further analysis, refine feature selection and modeling based on full metadata documentation. Data provenance and schema compliance is enhanced by referencing every entity by its Croissant `@id`.